In [ ]:
%pip install openai scikit-learn pandas

In [ ]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1", # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
# Set model parameters

SYSTEM_PROMPT = """
You a GAD-7 survey analyzer.
Your job is to extract answers for GAD-7 based on a given transcript. You are working with another chatbot that will generate followup.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a widely used self-administered diagnostic tool designed to screen for and assess the severity of generalized anxiety disorder (GAD).
GAD-7 question order:
q1: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
q2: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
q3: Over the last two weeks, how often have you been bothered by worrying too much about different things?
q4: Over the last two weeks, how often have you been bothered by trouble relaxing?
q5: Over the last two weeks, how often have you been bothered by being so restless that it is hard to sit still?
q6: Over the last two weeks, how often have you been bothered by becoming easily annoyed or irritable?
q7: Over the last two weeks, how often have you been bothered by feeling afraid, as if something awful might happen?
Scale mapping:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day


Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- For each question q1-q7, determine the score 0-3 based on the user's responses. Only score questions you are confident you can answer correctly. Otherwise, do not score the question and leave that key-value pair out of your answer.
- Return ONLY valid JSON. Empty object is acceptable null.
- Do not include markdown fences.
- Do not include explanation.
- Return JSON in exactly this shape and no other text:

{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
"""
MODEL = 'qwen3.5:9b'

- Use a json dataset of conversation transcript
- Extract scores using LLM or NLP technique
- Validate whether they were correct

In [ ]:
# Load the GAD dataset
import json 

with open('gad_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')

In [ ]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(len(conv_train), len(_conv_test), len(score_train), len(_score_test))

In [ ]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

In [ ]:
print(conv_train[0])

In [ ]:
def agent_loop(full_conversation):
    result = {
        "q1": "",
        "q2": "",
        "q3": "",
        "q4": "",
        "q5": "",
        "q6": "",
        "q7": ""
    }
    def get_result() -> str:
        """
        Get the current survey's scores as stringified JSON.
        """
        return str(result)

    def change_result(changed_scores: list[list[str, str]]):
        """
        Change the current survey's scores.
        Input is an array of key/value pairs passed in as tuples.
        Ex.
        To change q1 to score 3, input is `[("q1", "3")]`
        """
        for s in changed_scores:
            (k, v) = s
            if k in result.keys():
                result[k] = v
        
        return "success"
    
    def tool_handler(tool_name: str, args):
        if tool_name == "get_result":
            get_result(**args)
        elif tool_name == "change_result":
            change_result(**args)
        
    
    tool_get_result = {
        "type": "function",
        "name": "get_result",
        "description": "Get the current survey's scores as stringified JSON.",
        "parameters": {},
        "strict": True,
        "required": []
    }
    
    tool_change_result = {
        "type": "function",
        "name": "change_result",
        "description": """Change the current survey's scores. \
        Input is an array of key/value pairs passed in as arrays. \
        Example: To change q1 to score 3 and q5 to 1, input is `[["q1", "3"], ["q5", "1"]]`""",
        "parameters": {
            "type": "object",
            "properties": {
                "changed_scores": {
                    "type": "array",
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "string",
                        },
                        "minItems": 2,
                        "maxItems": 2,
                    }
                }
            }
        },
        "strict": True,
        "required": ["changed_scores"]
    }
    
    # Define the agent's tools
    tools = [
        tool_change_result,
        tool_get_result
    ]

    m = [Message(SYSTEM_PROMPT, 'system')]
    m.append(full_conversation[1]) # Should start at index 0 but dataset erroneously starts with assistant message

    # Simulate ask/response between chatbot and user from dataset
    for i in range(3, len(full_conversation), 2): # Should start at index 2 but dataset erroneously starts with assistant message
        
        # Invoke tools using agent
        response = client.chat.completions.create(
            model=MODEL,
            messages=m,
            tools=tools,
            tool_choice="auto",
        )

        message = response.choices[0].message

        # Handle tool calls
        for tool_call in message.tool_calls:
            result = tool_handler(tool_call.function.name, json.loads(tool_call.function.arguments))

            # Save function call outputs
            m.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": tool_call.function.name,
                "content": result
            })
        
        # Determine if we have completed the survey
        if "" not in result.values():
            break 

        # Add next assistant message and next user message to the context 
        m.extend(full_conversation[i-1:i])

    return result

scores = []
for i, c in enumerate(conv_train):
    print(f'Generating {i+1}/{len(conv_train)}')
    scores.append(agent_loop(c))
    print(scores[-1])

In [ ]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores):
    return [int(s) if s else 0 for s in scores.values.flatten().tolist()]

pred_df = pd.DataFrame(scores)
train_df = pd.DataFrame(score_train)

mse = mean_squared_error(
    convert_scores_to_array(train_df),
    convert_scores_to_array(pred_df)
)
print(mse) 

diff_df = train_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(columns={'self': 'Expected', 'other': 'Predicted'}, level=1)
display(diff_df)